In [45]:
import os
import numpy as np
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv


load_dotenv()

True

In [46]:
class Queries(BaseModel):
    queries: List[str] = Field(description="List of search queries")

In [34]:
json_dir = './json/content'
section_names = [x.replace('.json','') for x in os.listdir(json_dir) if '.json' in x]

docs = []
for i,section_name in enumerate(section_names):
    with open(f'{json_dir}/{section_name}.json', 'r') as f:
        # Parsing the JSON file into a Python dictionary
        content_list = json.load(f)
        for j,content in enumerate(content_list):
            text_content = f"Section: {section_name}\n\nTitle: {content['title']}\n\nContent: {content['content']}"
            metadata = {
                "section": section_name,
                "title": content['title'],
                "doc_id": f"section{i}-title{j}"
            }
            doc = Document(
                                page_content=text_content,
                                metadata=metadata
                            )
            docs.append(doc)

In [41]:
print(f"No. of docs: {len(docs)}")
np.random.seed(20)
rand_doc_i = np.random.randint(0,len(docs),10)
rand_doc_i

No. of docs: 200


array([ 99,  15, 156, 137, 148,  75,  22,  71, 162, 118])

In [48]:
llm = ChatOpenAI(model="gpt-4o")
llm_with_tool = llm.with_structured_output(Queries)

In [55]:
gen_question_prompt = """
Generate 3 questions that could be answered with the given document. 
If the document is not a good source to generate questions, please ask for a new document.

Context: The questions will be use to test the quality of RAG system.

Guidelines:
Be creative.
The questions must reflect the situation the users are facing, not quiz-type questions from a textbook.
Avoid using the same example of keywords or numbers from the document
"""

In [66]:
doc_questions = []
for i in rand_doc_i:
    doc = docs[i]
    doc_id = doc.metadata['doc_id']

    messages = [
                SystemMessage(content="You are a useful assistant of RAG system builder"),
                doc.page_content,
                HumanMessage(content=gen_question_prompt)
            ]
    gen_questions = llm_with_tool.invoke(messages)
    doc_question = {
        'doc_id':doc_id,
        'questions':gen_questions.queries
    }
    doc_questions.append(doc_question)

In [68]:
# Save to JSON
save_name = './json/test_questions.json'
with open(save_name, 'w', encoding='utf-8') as f:
    json.dump(doc_questions, f, indent=4, ensure_ascii=False)